# 02. Train / validation / test split

**Goal:** split the dataset chronologically in about 46% / 24% / 30%.

**Why chronological:** the real use case is predicting future videos from past ones.
A random split would let the model see the future and give overly optimistic scores.

- **Train:** early 2025 to early September 2025 (fits the model)
- **Validation:** September 2025 to January 2026 (tunes choices)
- **Test:** January 2026 to June 2026 (final honest evaluation, used once)

**Limitation:** the boundaries fall mid-month, so two months are shared by two splits.

In [1]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

df = pl.read_parquet(ROOT / "data" / "sample_recent" / "recent_60k_2025_26.parquet").sort("posted_at")
print(df.shape, df["posted_at"].min(), "->", df["posted_at"].max())

(59994, 31) 2025-01-01 00:21:32+00:00 -> 2026-06-30 23:09:45+00:00


split 46 / 24 / 30

In [2]:
n = df.height
n_train = int(n * 0.46)
n_val = int(n * 0.24)

train = df[:n_train]
val = df[n_train:n_train + n_val]
test = df[n_train + n_val:]

for name, d in [("train", train), ("validation", val), ("test", test)]:
    print(name, d.height, d["posted_at"].min(), "->", d["posted_at"].max())

train 27597 2025-01-01 00:21:32+00:00 -> 2025-09-04 18:43:02+00:00
validation 14398 2025-09-04 18:43:41+00:00 -> 2026-01-20 21:01:18+00:00
test 17999 2026-01-20 21:02:02+00:00 -> 2026-06-30 23:09:45+00:00


In [3]:
rows = []
for name, d in [("Train", train), ("Validation", val), ("Test", test)]:
    observed = d.filter((pl.col("width") > 0) & (pl.col("height") > 0)).height
    rows.append({
        "split": name,
        "Rows": d.height,
        "Dimensions_observed": observed,
        "Observed_share": round(observed / d.height, 6),
    })

summary = pl.DataFrame(rows)
print(summary)
summary.write_csv(ROOT / "reports" / "split_summary.csv")

shape: (3, 4)
┌────────────┬───────┬─────────────────────┬────────────────┐
│ split      ┆ Rows  ┆ Dimensions_observed ┆ Observed_share │
│ ---        ┆ ---   ┆ ---                 ┆ ---            │
│ str        ┆ i64   ┆ i64                 ┆ f64            │
╞════════════╪═══════╪═════════════════════╪════════════════╡
│ Train      ┆ 27597 ┆ 132                 ┆ 0.004783       │
│ Validation ┆ 14398 ┆ 101                 ┆ 0.007015       │
│ Test       ┆ 17999 ┆ 233                 ┆ 0.012945       │
└────────────┴───────┴─────────────────────┴────────────────┘


In [4]:
train.write_parquet(PROCESSED / "train.parquet")
val.write_parquet(PROCESSED / "validation.parquet")
test.write_parquet(PROCESSED / "test.parquet")

In [5]:
cols = {
    "country filled": (df["country"] != "").mean(),
    "language known": (df["language"] != "un").mean(),
    "caption filled": (df["caption"] != "").mean(),
    "has hashtags": (df["hashtags"].list.len() > 0).mean(),
    "has mentions": (df["mentions"].list.len() > 0).mean(),
    "has on_screen_text": (df["on_screen_text"].list.len() > 0).mean(),
    "duration > 0": (df["duration_ms"] > 0).mean(),
    "is_ad": df["is_ad"].mean(),
    "is_shop_video": df["is_shop_video"].mean(),
    "width > 0": (df["width"] > 0).mean(),
}
for k, v in cols.items():
    print(f"{k:20s} {v:.1%}")
print(df["type"].value_counts())

country filled       100.0%
language known       47.1%
caption filled       74.0%
has hashtags         56.9%
has mentions         8.1%
has on_screen_text   25.7%
duration > 0         92.9%
is_ad                2.2%
is_shop_video        3.9%
width > 0            0.8%
shape: (2, 2)
┌───────┬───────┐
│ type  ┆ count │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ video ┆ 55726 │
│ photo ┆ 4268  │
└───────┴───────┘
